# 02. Geração de Pseudo-Labels com DeepForest

Este notebook executa o modelo pré-treinado DeepForest (RetinaNet) sobre as imagens compactadas no arquivo HDF5, converte as caixas para o formato YOLO [class_id, x_center, y_center, w, h] normalizado, e salva as anotações geradas no subgrupo `labels` correspondente.

**Responsável:** Pessoa 3

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np

from deepforest import main

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

## 1. Carregamento do HDF5 e do Modelo DeepForest

In [ ]:
import pandas as pd

HDF5_PATH = PROJECT_ROOT / "data" / "dataset_v1_raw.h5"

hdf5 = h5py.File(HDF5_PATH, "r+")
images: h5py.Group = hdf5["images"]  # type: ignore[assignment]

print("Total de imagens:", len(images))

model = main.deepforest()
model.load_model()  # ← carrega do HuggingFace Hub

print("Modelo carregado com sucesso")

## 2. Predição de Caixas e Normalização YOLO (Pessoa 3)

Conversão de [xmin, ymin, xmax, ymax] em pixels para as coordenadas normalizadas [0.0, 1.0].

In [ ]:
def convert_to_yolo(box: list, img_w: int, img_h: int) -> list:
    """Converte [xmin, ymin, xmax, ymax] em pixels para YOLO normalizado."""
    xmin, ymin, xmax, ymax = box
    x_center = (xmin + xmax) / 2 / img_w
    y_center = (ymin + ymax) / 2 / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h
    return [x_center, y_center, width, height]


# Mapeia tile_name -> array Nx5 [class_id, x_center, y_center, w, h]
pseudo_labels: dict[str, np.ndarray] = {}

try:
    for tile_name in sorted(images.keys()):
        img_array: np.ndarray = images[tile_name][:]  # type: ignore[index]

        if img_array.dtype != np.uint8:
            img_array = img_array.astype(np.uint8)

        # HDF5 armazena em BGR (padrão OpenCV); DeepForest espera RGB
        img_rgb = img_array[..., ::-1]

        preds = model.predict_image(image=img_rgb)

        h, w = img_array.shape[:2]
        boxes: list = []

        if preds is not None and len(preds) > 0:
            for _, row in preds.iterrows():
                yolo_box = convert_to_yolo(
                    [row["xmin"], row["ymin"], row["xmax"], row["ymax"]], w, h
                )
                boxes.append([0] + yolo_box)  # class_id=0 (tree)

        pseudo_labels[tile_name] = np.array(boxes, dtype=np.float32).reshape(-1, 5)
        print(f"{tile_name}: {len(boxes)} árvores detectadas")

finally:
    pass  # hdf5 fechado na célula de gravação com try/finally

print(f"\nTotal de tiles processados: {len(pseudo_labels)}")
print(f"Total de pseudo-labels: {sum(len(v) for v in pseudo_labels.values())}")

## 3. Gravação das Pseudo-Labels no HDF5

In [ ]:
import utils

try:
    for tile_name, boxes in pseudo_labels.items():
        utils.salvar_pseudo_labels(str(HDF5_PATH), tile_name, boxes)

    print(f"Pseudo-labels salvas com sucesso no HDF5: {HDF5_PATH}")
    print(f"Tiles gravados: {len(pseudo_labels)}")
    print(f"Total de caixas: {sum(len(v) for v in pseudo_labels.values())}")
finally:
    hdf5.close()

## 4. Exportação para Roboflow (Pessoa 4)

Extrai imagens e labels do HDF5 para uma estrutura de pastas compatível com o Roboflow,
gerando também um `.zip` pronto para upload.

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "data" / "roboflow_export"

resultado = utils.exportar_hdf5_para_roboflow(
    hdf5_path=str(HDF5_PATH),
    output_dir=str(EXPORT_DIR),
)

print(f"Imagens exportadas : {resultado['imagens']}")
print(f"Caixas exportadas  : {resultado['labels']}")
print(f"Diretório          : {resultado['output_dir']}")
print(f"ZIP para upload    : {resultado['zip_path']}")